# 0. Load Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from dotenv import load_dotenv

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, precision_recall_curve)


# 1. Load data

In [21]:
sns.set_style('darkgrid')
np.random.seed(42)

env_file = '/content/env'

if os.path.exists(env_file):
    load_dotenv(env_file)

if not os.environ.get('KAGGLE_USERNAME') or not os.environ.get('KAGGLE_KEY'):
    raise RuntimeError(
        "Missing KAGGLE_USERNAME / KAGGLE_KEY. Create a .env file with:\n"
        "KAGGLE_USERNAME=your_username\nKAGGLE_KEY=your_key"
    )


In [22]:

dataset_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
csv_path = os.path.join(dataset_path, 'creditcard.csv')
print("Dataset downloaded to:", dataset_path)

df = pd.read_csv(csv_path)
print(df.shape)
print(df['Class'].value_counts(normalize=True) * 100)

df = df.rename(columns={
    "Amount_scaled": "new_name1",
    "Time_scaled": "new_name2"
})

fraud = df[df['Class'] == 1]
normal = df[df['Class'] == 0]
print(f"Fraud cases: {len(fraud)}  Normal cases: {len(normal)}")
print(f"Imbalance ratio: 1 : {round(len(normal)/len(fraud))}")

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Dataset downloaded to: /kaggle/input/creditcardfraud
(284807, 31)
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64
Fraud cases: 492  Normal cases: 284315
Imbalance ratio: 1 : 578


In [23]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


# 2. EDA

In [24]:
# fig, ax = plt.subplots(1, 2, figsize=(14, 4))
# sns.countplot(x='Class', data=df, ax=ax[0])
# ax[0].set_title('Class distribution')
# sns.histplot(df['Amount'], bins=50, ax=ax[1], kde=True)
# ax[1].set_title('Transaction amount distribution')
# plt.tight_layout()
# plt.savefig('class_distribution.png', dpi=120)
# plt.close()

# plt.figure(figsize=(10, 4))
# sns.boxplot(x='Class', y=np.log1p(df['Amount']), data=df)
# plt.title('Log-transformed amount by class')
# plt.ylabel('log(1 + Amount)')
# plt.savefig('amount_by_class.png', dpi=120)
# plt.close()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(x='Class', data=df, ax=ax[0])
ax[0].set_title('Class distribution')

sns.histplot(np.log1p(df['Amount']), bins=50, ax=ax[1], kde=True)
ax[1].set_title('Transaction amount distribution (log1p scale)')
ax[1].set_xlabel('log(1 + Amount)')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120)
plt.close()

# plt.figure(figsize=(10, 4))
# sns.boxplot(x='Class', y='Amount', data=df[df['Amount'] < 2500])
# plt.title('Amount by class (outliers > 2500 removed for readability)')
# plt.savefig('amount_by_class.png', dpi=120)
# plt.close()

plt.figure(figsize=(10, 4))
sns.boxplot(x='Class', y=np.log1p(df['Amount']), data=df)
plt.title('Log-transformed amount by class')
plt.ylabel('log(1 + Amount)')
plt.savefig('amount_by_class.png', dpi=120)
plt.close()

corr = df.corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.1)
plt.title('Feature correlation heatmap')
plt.savefig('correlation_heatmap.png', dpi=120)
plt.close()

# 3. Preprocessing

In [25]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])
df.drop(['Amount', 'Time'], axis=1, inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

contamination = y_train.sum() / len(y_train)
print(f"Estimated contamination: {contamination:.5f}")

Estimated contamination: 0.00173


# 4. Models

In [26]:
results = {}

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc*100:.2f}%")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraud']))
    print(confusion_matrix(y_true, y_pred))
    results[name] = acc
    return acc

In [27]:
# Isolation Forest
iso = IsolationForest(n_estimators=150, contamination=contamination,
                       max_samples=0.8, random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_pred = iso.predict(X_test)
iso_pred = np.where(iso_pred == -1, 1, 0)
evaluate('Isolation Forest', y_test, iso_pred)


--- Isolation Forest ---
Accuracy: 99.77%
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     85295
       Fraud       0.34      0.31      0.32       148

    accuracy                           1.00     85443
   macro avg       0.67      0.65      0.66     85443
weighted avg       1.00      1.00      1.00     85443

[[85204    91]
 [  102    46]]


0.997741184181267

In [29]:
# Local Outlier Factor (novelty mode so it can be used on unseen test data)
lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination, novelty=True, n_jobs=-1)
lof.fit(X_train)
lof_pred = lof.predict(X_test)
lof_pred = np.where(lof_pred == -1, 1, 0)
evaluate('Local Outlier Factor', y_test, lof_pred)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(



--- Local Outlier Factor ---
Accuracy: 99.66%
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     85295
       Fraud       0.00      0.00      0.00       148

    accuracy                           1.00     85443
   macro avg       0.50      0.50      0.50     85443
weighted avg       1.00      1.00      1.00     85443

[[85151   144]
 [  148     0]]


0.9965825169996372

In [31]:
# One-Class SVM — trained on a normal-only subsample since SVM doesn't scale well to 200k+ rows
svm_train = X_train[y_train == 0].sample(n=8000, random_state=42)
ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=contamination)
ocsvm.fit(svm_train)
svm_pred = ocsvm.predict(X_test)
svm_pred = np.where(svm_pred == -1, 1, 0)
evaluate('One-Class SVM', y_test, svm_pred)


--- One-Class SVM ---
Accuracy: 96.00%
              precision    recall  f1-score   support

      Normal       1.00      0.96      0.98     85295
       Fraud       0.03      0.83      0.07       148

    accuracy                           0.96     85443
   macro avg       0.52      0.90      0.52     85443
weighted avg       1.00      0.96      0.98     85443

[[81899  3396]
 [   25   123]]


0.9599616118347905

# 5. Isolation Forest with anomaly score based threshold tuning

In [32]:
scores = iso.decision_function(X_test)
precision, recall, thresholds = precision_recall_curve(y_test, -scores)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]
print(f"\nBest threshold on decision score: {best_threshold:.4f}, F1: {f1[best_idx]:.4f}")

tuned_pred = np.where(-scores >= best_threshold, 1, 0)
evaluate('Isolation Forest (tuned threshold)', y_test, tuned_pred)

auc = roc_auc_score(y_test, -scores)
print(f"Isolation Forest ROC-AUC: {auc:.4f}")


Best threshold on decision score: -0.0345, F1: 0.3677

--- Isolation Forest (tuned threshold) ---
Accuracy: 99.73%
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     85295
       Fraud       0.31      0.45      0.37       148

    accuracy                           1.00     85443
   macro avg       0.66      0.72      0.68     85443
weighted avg       1.00      1.00      1.00     85443

[[85150   145]
 [   82    66]]
Isolation Forest ROC-AUC: 0.9443


In [33]:
results

{'Isolation Forest': 0.997741184181267,
 'Local Outlier Factor': 0.9965825169996372,
 'One-Class SVM': 0.9599616118347905,
 'Isolation Forest (tuned threshold)': 0.9973432580784851}

# 6. PCA visualization of anomalies

In [34]:
pca = PCA(n_components=2, random_state=42)
X_test_pca = pca.fit_transform(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(X_test_pca[tuned_pred == 0, 0], X_test_pca[tuned_pred == 0, 1],
            s=5, alpha=0.4, label='Normal')
plt.scatter(X_test_pca[tuned_pred == 1, 0], X_test_pca[tuned_pred == 1, 1],
            s=15, color='red', label='Flagged as fraud')
plt.legend()
plt.title('Isolation Forest predictions in PCA space')
plt.savefig('pca_predictions.png', dpi=120)
plt.close()

# 7. Model comparison summary

In [35]:
print("\nModel comparison (accuracy):")
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v*100:.2f}%")

summary = pd.DataFrame(results.items(), columns=['Model', 'Accuracy'])
summary.to_csv('model_comparison.csv', index=False)

plt.figure(figsize=(8, 5))
sns.barplot(x='Accuracy', y='Model', data=summary.sort_values('Accuracy'))
plt.xlim(0.9, 1.0)
plt.title('Accuracy comparison across models')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120)
plt.close()


Model comparison (accuracy):
  Isolation Forest: 99.77%
  Isolation Forest (tuned threshold): 99.73%
  Local Outlier Factor: 99.66%
  One-Class SVM: 96.00%
